# Cross Tracker

In [1]:
import math
import random
import time
from typing import Tuple, List

import gymnasium
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3.common.vec_env import DummyVecEnv


## This is for when the episode ends after there is a collision

In [ ]:
from enum import Enum
import gymnasium as gym
from gymnasium import spaces
import pygame
import numpy as np
import math
import random
from typing import Tuple, List

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 8}

    def __init__(self,
                 grid_size: Tuple[int,int] = (64,64),
                 v_max: float = 0.6,
                 omega_max: float = 2.0,
                 max_steps: int = 500,
                 render_mode: str = None,
                 seed: int = None):
        super().__init__()
        self.grid_w, self.grid_h = grid_size
        self.v_max = v_max
        self.omega_max = omega_max
        self.max_steps = max_steps
        self.render_mode = render_mode
        self.window_size = 512

        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, -math.pi], dtype=np.float32),
            high=np.array([1.0, 1.0, math.pi], dtype=np.float32),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=np.array([0.0, -self.omega_max], dtype=np.float32),
            high=np.array([self.v_max, self.omega_max], dtype=np.float32),
            dtype=np.float32
        )

        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles: List[Tuple[int,int,int,int]] = []
        self._build_default_map()

        self.window = None
        self.clock = None

        self.seed(seed)

    def seed(self, s=None):
        self.np_random, seed = gym.utils.seeding.np_random(s)
        random.seed(int(seed % (2**32 - 1)))
        np.random.seed(int(seed % (2**32 - 1)))
        return [int(seed % (2**32 - 1))]

    def _build_default_map(self):
        self.grid_w, self.grid_h = 8, 8
        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles = []
    
        # --- Fixed obstacle layout ---
        layout = np.array([
            [0,0,1,0,0,0,0,0],
            [0,0,1,0,1,1,1,0],
            [0,0,0,0,1,0,1,0],
            [1,0,0,0,1,0,0,0],
            [1,0,0,1,0,0,1,1],
            [1,1,0,0,0,0,0,0],
            [0,0,0,0,1,1,1,1],
            [0,0,0,1,1,1,1,1],   
        ], dtype=np.uint8)
    
        self.occupancy = layout.copy()
    
        for y in range(self.grid_h):
            for x in range(self.grid_w):
                if self.occupancy[y, x] == 1:
                    self._add_obstacle(x, y, x+1, y+1)
    
        self._reserved_start = (0, 7)   
        self._reserved_goal = (5, 3)    

    def _add_obstacle(self, x1, y1, x2, y2):
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(self.grid_w, x2), min(self.grid_h, y2)
        if x2 > x1 and y2 > y1:
            self.occupancy[y1:y2, x1:x2] = 1
            self.obstacles.append((x1, y1, x2, y2))

    def _grid_to_world(self, gx, gy) -> Tuple[float,float]:
        return ( (gx + 0.5) / self.grid_w, (gy + 0.5) / self.grid_h )

    def _world_to_grid(self, x, y) -> Tuple[int,int]:
        gx = min(self.grid_w-1, max(0, int(math.floor(x * self.grid_w))))
        gy = min(self.grid_h-1, max(0, int(math.floor(y * self.grid_h))))
        return gx, gy

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if hasattr(self, "_reserved_start") and hasattr(self, "_reserved_goal"):
            sx, sy = self._grid_to_world(*self._reserved_start)
            gx, gy = self._grid_to_world(*self._reserved_goal)
        else:
            free_indices = np.argwhere(self.occupancy == 0)
            start_idx = free_indices[self.np_random.choice(len(free_indices))]
            goal_idx = free_indices[self.np_random.choice(len(free_indices))]
            sx, sy = self._grid_to_world(start_idx[1], start_idx[0])
            gx, gy = self._grid_to_world(goal_idx[1], goal_idx[0])

        self.state = np.array([sx, sy], dtype=np.float32)
        self.theta = random.uniform(-math.pi, math.pi)
        self.goal = np.array([gx, gy], dtype=np.float32)
        self.goal_radius = 0.08
        self.step_count = 0
        self.last_action = np.zeros(2, dtype=np.float32)

        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        info = {"distance": float(np.linalg.norm(self.state - self.goal))}
        if self.render_mode == "human":
            self._render_frame()
        return obs, info

    def _compute_obstacle_penalty(self, pos: np.ndarray) -> float:
        gx, gy = self._world_to_grid(pos[0], pos[1])
        search = 8
        gx0, gx1 = max(0, gx - search), min(self.grid_w - 1, gx + search)
        gy0, gy1 = max(0, gy - search), min(self.grid_h - 1, gy + search)
        patch = self.occupancy[gy0:gy1+1, gx0:gx1+1]
        if patch.sum() == 0:                               #velocity obstacle (VO)
                                                     ## add complexities - like when obstacles are also moving
                                                     ## passing through a window                        
            return 0.0
        ys, xs = np.where(patch == 1)
        xs, ys = xs + gx0, ys + gy0
        dists = np.sqrt((xs - gx) ** 2 + (ys - gy) ** 2)
        min_d = float(dists.min())
        return max(0.0, (search - min_d) / search)

    def step(self, action: np.ndarray):
        action = np.clip(action, self.action_space.low, self.action_space.high).astype(np.float32)
        v_cmd = (action[0] + 1.0) / 2 * self.v_max  
        omega_cmd = action[1] * self.omega_max       
    
        dt = 0.3
        self.theta += float(omega_cmd) * dt
        dx = float(v_cmd) * math.cos(self.theta) * dt
        dy = float(v_cmd) * math.sin(self.theta) * dt
    
        prev_dist = np.linalg.norm(self.state - self.goal)
        new_pos = np.clip(self.state + np.array([dx, dy], dtype=np.float32), 0.0, 1.0)
    
        gx, gy = self._world_to_grid(new_pos[0], new_pos[1])
        collision = bool(self.occupancy[gy, gx] == 1)
    
        self.state = new_pos
        new_dist = np.linalg.norm(self.state - self.goal)
    
        progress = (prev_dist - new_dist) * 50.0    
        obstacle_penalty = -1.5 * self._compute_obstacle_penalty(self.state)
        smooth_penalty = -0.1 * float(np.linalg.norm(action - self.last_action))
    
        reward = progress + obstacle_penalty + smooth_penalty
        self.last_action = np.copy(action)
        self.step_count += 1
    
        terminated = False
        truncated = False
        info = {}
    
        if collision:
            terminated = True
            reward = -10.0
            info = {"reason": "collision"}
        elif new_dist <= self.goal_radius:
            terminated = True
            reward += 100.0
            info = {"reason": "goal"}
            print("Goal reached!")
        elif self.step_count >= self.max_steps:
            truncated = True
            info = {"reason": "timeout"}
    
        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        return obs, float(reward), terminated, truncated, info


    def render(self):
        if self.render_mode == "rgb_array":
            return self._render_frame()
        elif self.render_mode == "human":
            self._render_frame()
        else:
            return None

    def _render_frame(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()
    
        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_w = self.window_size / self.grid_w
        pix_h = self.window_size / self.grid_h
    
        obs_coords = np.argwhere(self.occupancy == 1)
        for (gy, gx) in obs_coords:
            rect = pygame.Rect(
                int(gx * pix_w),
                int(self.window_size - (gy + 1) * pix_h),
                int(pix_w) + 1,
                int(pix_h) + 1
            )
            pygame.draw.rect(canvas, (0, 0, 0), rect)
    
        gx_f = int(self.goal[0] * self.window_size)
        gy_f = int(self.window_size - self.goal[1] * self.window_size)
        goal_size = int(min(pix_w, pix_h) * 0.8)
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(gx_f - goal_size // 2, gy_f - goal_size // 2, goal_size, goal_size)
        )
    
        ax = int(self.state[0] * self.window_size)
        ay = int(self.window_size - self.state[1] * self.window_size)
        pygame.draw.circle(canvas, (0, 0, 255), (ax, ay), int(min(pix_w, pix_h) * 0.4))
    
        for x in range(self.grid_w + 1):
            pygame.draw.line(canvas, (200, 200, 200), (int(x * pix_w), 0), (int(x * pix_w), self.window_size), 1)
        for y in range(self.grid_h + 1):
            y_pix = int(self.window_size - y * pix_h)
            pygame.draw.line(canvas, (200, 200, 200), (0, y_pix), (self.window_size, y_pix), 1)
    
        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])
        else:
            arr = pygame.surfarray.array3d(canvas)
            return np.transpose(arr, (1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None
            self.clock = None


## PPO

In [2]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
import numpy as np

def make_train_env():
    return GridWorldEnv(render_mode=None)

train_env = DummyVecEnv([make_train_env])  

check_env(GridWorldEnv(render_mode=None), warn=True)

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    tensorboard_log=None,
)

print("Starting training...")
model.learn(total_timesteps=10000)
print("Training finished. Saving model...")
model.save("ppo_crosstracker")

eval_env = GridWorldEnv(render_mode="human")  
obs, info = eval_env.reset()

for step in range(200): 
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()
train_env.close()


c:\Users\snigd\miniconda3\envs\myenv\lib\site-packages\stable_baselines3\common\env_checker.py:462: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


Using cpu device
Starting training...
Goal reached!
Goal reached!
-----------------------------
| time/              |      |
|    fps             | 1455 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 599         |
|    iterations           | 2           |
|    time_elapsed         | 6           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009744103 |
|    clip_fraction        | 0.113       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
|    explained_variance   | 0.00499     |
|    learning_rate        | 0.0003      |
|    loss                 | 23.7        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0126     |
|    std                  | 0.989       |
|    value

## DDPG

In [8]:
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env
import numpy as np

def make_train_env():
    return GridWorldEnv(render_mode=None)  
train_env = DummyVecEnv([make_train_env])
check_env(GridWorldEnv(render_mode=None), warn=True)

n_actions = train_env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = DDPG(
    "MlpPolicy",
    train_env,
    action_noise=action_noise,
    verbose=1,
    learning_rate=3e-4,
    gamma=0.99,
    batch_size=64,
    buffer_size=50000,
    tensorboard_log=None,
)

model.learn(total_timesteps=1000)
model.save("ddpg_gridworld")

eval_env = GridWorldEnv(render_mode="human")  
obs, info = eval_env.reset()

for _ in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()


Using cpu device
----------------------------
| time/              |     |
|    episodes        | 4   |
|    fps             | 704 |
|    time_elapsed    | 0   |
|    total_timesteps | 57  |
----------------------------
Goal reached!
----------------------------
| time/              |     |
|    episodes        | 8   |
|    fps             | 799 |
|    time_elapsed    | 0   |
|    total_timesteps | 96  |
----------------------------
---------------------------------
| time/              |          |
|    episodes        | 12       |
|    fps             | 300      |
|    time_elapsed    | 0        |
|    total_timesteps | 108      |
| train/             |          |
|    actor_loss      | 0.262    |
|    critic_loss     | 17.1     |
|    learning_rate   | 0.0003   |
|    n_updates       | 7        |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 16       |
|    fps             | 90       |
|    time_elapsed  

## SAC

In [ ]:
from enum import Enum
import gymnasium as gym
from gymnasium import spaces
import pygame
import numpy as np
import math
import random
from typing import Tuple, List

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 8}

    def __init__(self,
                 grid_size: Tuple[int,int] = (64,64),
                 v_max: float = 0.6,
                 omega_max: float = 2.0,
                 max_steps: int = 500,
                 render_mode: str = None,
                 seed: int = None):
        super().__init__()
        self.grid_w, self.grid_h = grid_size
        self.v_max = v_max
        self.omega_max = omega_max
        self.max_steps = max_steps
        self.render_mode = render_mode
        self.window_size = 512

        self.observation_space = spaces.Box(
            low=np.array([-1.0, -1.0, -math.pi], dtype=np.float32),
            high=np.array([1.0, 1.0, math.pi], dtype=np.float32),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=np.array([0.0, -self.omega_max], dtype=np.float32),
            high=np.array([self.v_max, self.omega_max], dtype=np.float32),
            dtype=np.float32
        )

        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles: List[Tuple[int,int,int,int]] = []
        self._build_default_map()

        self.window = None
        self.clock = None

        self.seed(seed)

    def seed(self, s=None):
        self.np_random, seed = gym.utils.seeding.np_random(s)
        random.seed(int(seed % (2**32 - 1)))
        np.random.seed(int(seed % (2**32 - 1)))
        return [int(seed % (2**32 - 1))]

    def _build_default_map(self):
        self.grid_w, self.grid_h = 8, 8
        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles = []
    
        # --- Fixed obstacle layout ---
        layout = np.array([
            [0,0,1,0,0,0,0,0],
            [0,0,1,0,1,1,1,0],
            [0,0,0,0,1,0,1,0],
            [1,0,0,0,1,0,0,0],
            [1,0,0,1,0,0,1,1],
            [1,1,0,0,0,0,0,0],
            [0,0,0,0,1,1,1,1],
            [0,0,0,1,1,1,1,1],   
        ], dtype=np.uint8)
    
        self.occupancy = layout.copy()
    
        for y in range(self.grid_h):
            for x in range(self.grid_w):
                if self.occupancy[y, x] == 1:
                    self._add_obstacle(x, y, x+1, y+1)
    
        self._reserved_start = (0, 7)   
        self._reserved_goal = (5, 3)    

    def _add_obstacle(self, x1, y1, x2, y2):
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(self.grid_w, x2), min(self.grid_h, y2)
        if x2 > x1 and y2 > y1:
            self.occupancy[y1:y2, x1:x2] = 1
            self.obstacles.append((x1, y1, x2, y2))

    # coordinate transforms
    def _grid_to_world(self, gx, gy) -> Tuple[float,float]:
        return ( (gx + 0.5) / self.grid_w, (gy + 0.5) / self.grid_h )

    def _world_to_grid(self, x, y) -> Tuple[int,int]:
        gx = min(self.grid_w-1, max(0, int(math.floor(x * self.grid_w))))
        gy = min(self.grid_h-1, max(0, int(math.floor(y * self.grid_h))))
        return gx, gy

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if hasattr(self, "_reserved_start") and hasattr(self, "_reserved_goal"):
            sx, sy = self._grid_to_world(*self._reserved_start)
            gx, gy = self._grid_to_world(*self._reserved_goal)
        else:
            free_indices = np.argwhere(self.occupancy == 0)
            start_idx = free_indices[self.np_random.choice(len(free_indices))]
            goal_idx = free_indices[self.np_random.choice(len(free_indices))]
            sx, sy = self._grid_to_world(start_idx[1], start_idx[0])
            gx, gy = self._grid_to_world(goal_idx[1], goal_idx[0])

        self.state = np.array([sx, sy], dtype=np.float32)
        self.theta = random.uniform(-math.pi, math.pi)
        self.goal = np.array([gx, gy], dtype=np.float32)
        self.goal_radius = 0.08
        self.step_count = 0
        self.last_action = np.zeros(2, dtype=np.float32)

        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        info = {"distance": float(np.linalg.norm(self.state - self.goal))}
        if self.render_mode == "human":
            self._render_frame()
        return obs, info

    def _compute_obstacle_penalty(self, pos: np.ndarray) -> float:
        gx, gy = self._world_to_grid(pos[0], pos[1])
        search = 8
        gx0, gx1 = max(0, gx - search), min(self.grid_w - 1, gx + search)
        gy0, gy1 = max(0, gy - search), min(self.grid_h - 1, gy + search)
        patch = self.occupancy[gy0:gy1+1, gx0:gx1+1]
        if patch.sum() == 0:
            return 0.0
        ys, xs = np.where(patch == 1)
        xs, ys = xs + gx0, ys + gy0
        dists = np.sqrt((xs - gx) ** 2 + (ys - gy) ** 2)
        min_d = float(dists.min())
        return max(0.0, (search - min_d) / search)

    def step(self, action: np.ndarray):
        action = np.clip(action, self.action_space.low, self.action_space.high).astype(np.float32)
        v_cmd = (action[0] + 1.0) / 2 * self.v_max   
        omega_cmd = action[1] * self.omega_max       
    
        dt = 0.3
        self.theta += float(omega_cmd) * dt
        dx = float(v_cmd) * math.cos(self.theta) * dt
        dy = float(v_cmd) * math.sin(self.theta) * dt
    
        prev_dist = np.linalg.norm(self.state - self.goal)
        new_pos = np.clip(self.state + np.array([dx, dy], dtype=np.float32), 0.0, 1.0)
    
        gx, gy = self._world_to_grid(new_pos[0], new_pos[1])
        collision = bool(self.occupancy[gy, gx] == 1)
    
        self.state = new_pos
        new_dist = np.linalg.norm(self.state - self.goal)
    

        progress = (prev_dist - new_dist) * 50.0    
        obstacle_penalty = -1.5 * self._compute_obstacle_penalty(self.state)
        smooth_penalty = -0.1 * float(np.linalg.norm(action - self.last_action))
    
        reward = progress + obstacle_penalty + smooth_penalty
        self.last_action = np.copy(action)
        self.step_count += 1
    
        terminated = False
        truncated = False
        info = {}
    
        if collision:
            terminated = True
            reward = -10.0
            info = {"reason": "collision"}
        elif new_dist <= self.goal_radius:
            terminated = True
            reward += 100.0
            info = {"reason": "goal"}
            print("Goal reached!")
        elif self.step_count >= self.max_steps:
            truncated = True
            info = {"reason": "timeout"}
    
        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        return obs, float(reward), terminated, truncated, info


    # Rendering via pygame (also supports rgb_array)
    def render(self):
        if self.render_mode == "rgb_array":
            return self._render_frame()
        elif self.render_mode == "human":
            self._render_frame()
        else:
            return None

    def _render_frame(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()
    
        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_w = self.window_size / self.grid_w
        pix_h = self.window_size / self.grid_h
    
        # Draw obstacles
        obs_coords = np.argwhere(self.occupancy == 1)
        for (gy, gx) in obs_coords:
            rect = pygame.Rect(
                int(gx * pix_w),
                int(self.window_size - (gy + 1) * pix_h),
                int(pix_w) + 1,
                int(pix_h) + 1
            )
            pygame.draw.rect(canvas, (0, 0, 0), rect)
    
        # Draw goal
        gx_f = int(self.goal[0] * self.window_size)
        gy_f = int(self.window_size - self.goal[1] * self.window_size)
        goal_size = int(min(pix_w, pix_h) * 0.8)
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(gx_f - goal_size // 2, gy_f - goal_size // 2, goal_size, goal_size)
        )
    
        # Draw agent
        ax = int(self.state[0] * self.window_size)
        ay = int(self.window_size - self.state[1] * self.window_size)
        pygame.draw.circle(canvas, (0, 0, 255), (ax, ay), int(min(pix_w, pix_h) * 0.4))
    
        # Draw grid lines
        for x in range(self.grid_w + 1):
            pygame.draw.line(canvas, (200, 200, 200), (int(x * pix_w), 0), (int(x * pix_w), self.window_size), 1)
        for y in range(self.grid_h + 1):
            y_pix = int(self.window_size - y * pix_h)
            pygame.draw.line(canvas, (200, 200, 200), (0, y_pix), (self.window_size, y_pix), 1)
    
        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])
        else:
            arr = pygame.surfarray.array3d(canvas)
            return np.transpose(arr, (1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None
            self.clock = None


In [16]:
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv

def make_env():
    return GridWorldEnv(render_mode="human")

train_env = DummyVecEnv([make_env])

check_env(GridWorldEnv(render_mode=None), warn=True)


model = SAC(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    gamma=0.99,
    batch_size=256,
    buffer_size=50000,
    tensorboard_log=None,
)

model.learn(total_timesteps=20000)
model.save("sac_gridworld")

eval_env = GridWorldEnv(render_mode="human")  
obs, info = eval_env.reset()

for _ in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()


Using cpu device
---------------------------
| time/              |    |
|    episodes        | 4  |
|    fps             | 67 |
|    time_elapsed    | 0  |
|    total_timesteps | 43 |
---------------------------
---------------------------
| time/              |    |
|    episodes        | 8  |
|    fps             | 82 |
|    time_elapsed    | 1  |
|    total_timesteps | 94 |
---------------------------
---------------------------------
| time/              |          |
|    episodes        | 12       |
|    fps             | 61       |
|    time_elapsed    | 2        |
|    total_timesteps | 143      |
| train/             |          |
|    actor_loss      | -1.06    |
|    critic_loss     | 7.45     |
|    ent_coef        | 0.988    |
|    ent_coef_loss   | -0.0416  |
|    learning_rate   | 0.0003   |
|    n_updates       | 42       |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 16       |
|    fps     

KeyboardInterrupt: 

## TD3

In [17]:
from stable_baselines3 import TD3
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv

train_env = DummyVecEnv([make_env])
check_env(GridWorldEnv(render_mode=None), warn=True)

model = TD3(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    gamma=0.99,
    batch_size=256,
    buffer_size=100000,
    tensorboard_log=None,
)

model.learn(total_timesteps=10000)
model.save("td3_gridworld")

for _ in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()

Using cpu device
---------------------------
| time/              |    |
|    episodes        | 4  |
|    fps             | 60 |
|    time_elapsed    | 0  |
|    total_timesteps | 39 |
---------------------------
---------------------------
| time/              |    |
|    episodes        | 8  |
|    fps             | 72 |
|    time_elapsed    | 1  |
|    total_timesteps | 83 |
---------------------------
---------------------------------
| time/              |          |
|    episodes        | 12       |
|    fps             | 58       |
|    time_elapsed    | 1        |
|    total_timesteps | 110      |
| train/             |          |
|    actor_loss      | 1.36     |
|    critic_loss     | 24.7     |
|    learning_rate   | 0.0003   |
|    n_updates       | 9        |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 16       |
|    fps             | 49       |
|    time_elapsed    | 2        |
|    total_ti